In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Iniciar a SparkSession com configurações otimizadas
spark = SparkSession.builder \
    .appName("Transformação Data Silver") \
    .config("spark.sql.shuffle.partitions", "200")  \
    .config("spark.sql.files.maxPartitionBytes", "128MB") \
    .config("spark.sql.parquet.compression.codec", "snappy") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

In [0]:
dbutils.fs.mkdirs("/Volumes/vendas_pecas/default/my/prata")

In [0]:
bronze_path = "/Volumes/vendas_pecas/default/my/bronze"
prata_path = "/Volumes/vendas_pecas/default/my/prata"

In [0]:
df_bronze = spark.read.format("parquet").load(bronze_path)


In [0]:
from pyspark.sql.functions import format_number

#normalizando valores decimais e alterando o nome da coluna IdNf para melhor compreendimento

df_silver = df_bronze.withColumnRenamed("id_nf", "IdNotaFiscal") \
                     .withColumn("valor_unitario", col("valor_unitario").cast(DecimalType(18, 2)))\
                     .withColumn("custo_unitario", col("custo_unitario").cast(DecimalType(18, 2)))

In [0]:
from pyspark.sql.functions import translate,lower
acentos     = "áàãâÁÀÃÂéèêÉÈÊíìîÍÌÎóòõôÓÒÕÔúùûÚÙÛçÇ"
sem_acentos = "aaaaAAAAeeeEEEiiiIIIooooOOOOuuuUUUcC"
#removendo acentos da tabela produto_peca
df_silver = df_silver.withColumn("produto_peca_sem_acento", translate(df_silver.produto_peca, acentos, sem_acentos))\
                     .withColumn("loja_nome_sem_acento", translate(df_silver.loja_nome, acentos, sem_acentos))\
                     .withColumn("grupo_loja_sem_acento", translate(df_silver.grupo_loja, acentos, sem_acentos))

In [0]:
df_silver.withColumn("data_venda", to_date(col("data_venda"), "yyyy-MM-dd"))

In [0]:
#adicionando colunas ano e mes de acordo com a coluna data ja existente  na tabela
df_silver = df_silver.withColumn("ano_venda", year(col("data_venda")))\
                     .withColumn("mes_venda", month(col("data_venda")))

In [0]:
df_silver = df_silver.dropDuplicates(["IdNotaFiscal"])


In [0]:
df_silver.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable("vendas_pecas.camada_prata.tb_prata")

In [0]:
df = spark.table('vendas_pecas.camada_prata.tb_prata')

if 'loja_uf' not in df.columns:
    spark.sql(f'ALTER TABLE vendas_pecas.camada_prata.tb_prata ADD COLUMN loja_uf STRING')

In [0]:
%sql
UPDATE vendas_pecas.camada_prata.tb_prata
SET loja_uf =
CASE
  WHEN CAST (loja_id AS STRING) LIKE '100%' THEN 'SP'
  WHEN CAST (loja_id AS STRING) LIKE '200%' THEN 'RJ'
  WHEN CAST (loja_id AS STRING) LIKE '300%' THEN 'MG'
  ELSE 'N/A'
END           